# Cross-modal sanity check: TERRA + UNI2-h on TENX99

Goal: compare three linear-probe baselines for gene expression prediction on a
subset of TENX99 cells — TERRA embeddings alone, UNI2-h embeddings alone, and
their concatenation — to test whether TERRA (spatial transcriptomics FM) and
UNI2-h (pathology FM) carry complementary information. This notebook covers
**Steps 1-2** (cell subset + TERRA embeddings); UNI2-h extraction and the
linear probes follow in a later pass.

## A key fact this notebook works around

Per `01_explore_hest_xenium.ipynb`, HEST's shipped `TENX99.h5ad` is **not**
single-cell — it's pooled onto a ~100 um pseudo-spot grid (7,233 rows, vs.
365,604 true Xenium cells). TERRA is a single-cell model (its own quickstarts
run on true single-cell Xenium `AnnData`), and "sample 10,000 cells" isn't
possible from a 7,233-row pooled object anyway. So **Step 1 reconstructs a
true single-cell `AnnData` for TENX99 from the raw per-transcript calls**
(`transcripts/TENX99_transcripts.parquet`), using exactly the `cell_id`-based
transcript-to-cell linking shown in the task brief. Cell centroids are the
mean `he_x`/`he_y` of each cell's transcripts, rather than `xenium_seg`'s
segmentation polygons -- `xenium_seg`'s parquet has no explicit `cell_id`
column, only row order, which we found doesn't reliably match the `cell_id`
values in `transcripts.parquet`.

## Environment note

This notebook runs in the **terra-dev** kernel (has `terra-st`, `torch`+CUDA,
`squidpy`, `geopandas`, `datasets`, `transformers` — everything needed for
TERRA and, later, UNI2-h). It does **not** have the HEST-Library/OpenSlide
stack used in `01_explore_hest_xenium.ipynb`, so HEST's parquet/segmentation
files are read directly with pandas/geopandas here rather than through
`HESTData`/`iter_hest` — same files, no HEST-Library dependency needed for
what this notebook does.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix

HEST_DIR = Path("/lustre/scratch126/cellgen/lotfollahi/so25/hest_data")
NOTEBOOK_DIR = Path("/lustre/scratch126/cellgen/lotfollahi/so25/terra-multi/notebooks")
DATA_DIR = NOTEBOOK_DIR.parent / "data"
EMB_DIR = DATA_DIR / "embeddings"
CACHE_DIR = DATA_DIR / "terra_cache"
EMB_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_ID = "TENX99"
RANDOM_STATE = 42
N_CELLS = 10_000
MIN_TOTAL_COUNTS = 50

plt.rcParams["figure.dpi"] = 110

## Step 1 — Select cell subset

### 1a. Build a true single-cell AnnData for TENX99

We read only the columns we need from the 53.6M-row transcripts file
(`cell_id`, `feature_name`, `he_x`, `he_y`), drop unassigned transcripts
(`cell_id == -1`, Xenium's convention for calls not attributed to any cell)
and Xenium's negative-control probes (`NegControlProbe_*`,
`NegControlCodeword_*`, `BLANK_*` — see the note in
`01_explore_hest_xenium.ipynb` about these being miscounted as genes there),
then aggregate transcript counts per (cell, gene) into a sparse cells x genes
matrix. Cell centroids are the mean `he_x`/`he_y` position of each cell's
transcripts -- `he_x`/`he_y` are already non-rigidly registered onto the H&E
(per notebook 1), and `cell_id` is an explicit column here, so this needs no
positional-index assumption the way joining against `xenium_seg`'s
geometry-only parquet would.

In [ ]:
def is_control_probe(name):
    return name.startswith(("NegControlProbe_", "NegControlCodeword_", "BLANK_"))


transcripts = pd.read_parquet(
    HEST_DIR / "transcripts" / f"{SAMPLE_ID}_transcripts.parquet",
    columns=["cell_id", "feature_name", "he_x", "he_y"],
)
print(f"Raw transcript rows: {len(transcripts):,}")

transcripts = transcripts[transcripts["cell_id"] != -1]
print(f"After dropping unassigned (cell_id == -1): {len(transcripts):,}")

transcripts = transcripts[
    ~transcripts["feature_name"].astype(str).map(is_control_probe)
]
print(f"After dropping negative-control probes: {len(transcripts):,}")
print(f"Real genes remaining: {transcripts['feature_name'].nunique()}")

In [ ]:
# Per-cell centroid: mean he_x/he_y of that cell's transcripts. he_x/he_y are
# already non-rigidly registered onto the H&E (confirmed in notebook 1), and
# cell_id is an explicit column here -- unlike xenium_seg's parquet (geometry
# only, no cell_id), so this avoids assuming row position == cell_id (an
# assumption that turned out to be wrong when we tried it).
cell_centroids = transcripts.groupby("cell_id")[["he_x", "he_y"]].mean()
cell_centroids.columns = ["cx", "cy"]
print(f"Centroids computed for {len(cell_centroids):,} cells")

In [ ]:
# Aggregate transcript counts into a sparse (cells x genes) matrix
cell_cat = pd.Categorical(transcripts["cell_id"])
gene_cat = pd.Categorical(transcripts["feature_name"])

counts = (
    coo_matrix(
        (np.ones(len(transcripts), dtype=np.float32), (cell_cat.codes, gene_cat.codes))
    ).tocsr()
)  # duplicate (cell, gene) coordinates are summed on conversion -> transcript counts

adata_full = ad.AnnData(
    X=counts,
    obs=pd.DataFrame(index=[str(c) for c in cell_cat.categories]),
    var=pd.DataFrame(index=list(gene_cat.categories)),
)
adata_full.obs["cell_id"] = adata_full.obs_names
adata_full.obsm["spatial"] = cell_centroids.loc[
    cell_cat.categories, ["cx", "cy"]
].to_numpy()

sc.pp.calculate_qc_metrics(adata_full, inplace=True, percent_top=None)
print(adata_full)

### 1b. Filter and sample 10,000 cells

Drop cells with `total_counts < 50` (very low-count cells are usually noise),
then sample N=10,000 uniformly at random with `random_state=42`. The filtered
population is far larger than 10,000, so a single `sample()` call always
yields exactly 10,000 — the "resample if needed" contingency in the brief
doesn't trigger here, but we assert on it rather than assume.

In [ ]:
n_before = adata_full.n_obs
adata_ok = adata_full[adata_full.obs["total_counts"] >= MIN_TOTAL_COUNTS].copy()
n_after = adata_ok.n_obs
print(f"Cells before filtering: {n_before:,}")
print(f"Cells after total_counts >= {MIN_TOTAL_COUNTS}: {n_after:,}")
print(f"Dropped: {n_before - n_after:,} ({100 * (n_before - n_after) / n_before:.1f}%)")

assert n_after >= N_CELLS, (
    f"Only {n_after} cells pass the filter -- fewer than the requested {N_CELLS}; "
    "would need to sample with replacement or lower N_CELLS."
)

sampled_obs = adata_ok.obs.sample(n=N_CELLS, random_state=RANDOM_STATE)
sampled_cell_ids = sampled_obs["cell_id"].to_numpy()
assert len(set(sampled_cell_ids)) == N_CELLS
print(f"Sampled exactly {len(sampled_cell_ids):,} cells (random_state={RANDOM_STATE})")

In [ ]:
# Distribution of total_counts in the sampled subset
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(
    adata_ok.obs["total_counts"], bins=60, color="lightgray", label="all filtered cells"
)
axes[0].hist(
    sampled_obs["total_counts"],
    bins=60,
    color="steelblue",
    alpha=0.7,
    label="sampled 10k",
)
axes[0].set_title("total_counts: filtered population vs sampled subset")
axes[0].set_xlabel("total_counts")
axes[0].legend()

axes[1].hist(sampled_obs["total_counts"], bins=60, color="steelblue")
axes[1].axvline(
    sampled_obs["total_counts"].median(), color="black", linestyle="--", label="median"
)
axes[1].set_title("Sampled 10k: total_counts distribution")
axes[1].set_xlabel("total_counts")
axes[1].legend()
fig.tight_layout()

print(sampled_obs["total_counts"].describe())

In [ ]:
# Save the selected cell IDs now, independent of the embedding step below
np.save(EMB_DIR / f"{SAMPLE_ID}_sampled_cell_ids.npy", sampled_cell_ids)
print(
    f"Saved {len(sampled_cell_ids):,} cell IDs -> {EMB_DIR / f'{SAMPLE_ID}_sampled_cell_ids.npy'}"
)

## Step 2 — Extract TERRA embeddings

### What we learned from `/lustre/scratch126/cellgen/lotfollahi/so25/notebooks`

The quickstart/downstream notebooks there establish the actual TERRA API:

- `download_pretrained("Lotfollahi-lab/TERRA-112M")` fetches a self-contained
  model bundle (checkpoint, tokenizer, gene-reference files).
- The single-call convenience path is `harmonize_tokenize_embed_pipeline`, but
  it always embeds **every** cell passed in — no subsetting. For "compute
  neighbors on the full section, then only forward the 10,000 index cells"
  we need its three underlying steps (all public API: `harmonize_adata`,
  `tokenize_adata`, `embed_dataset`), tokenizing the full section once and
  then restricting the tokenized dataset before the (expensive, GPU) embed
  step — exactly the pattern `downstream_analysis.ipynb`'s "Analyze a subset
  of cells" section uses (`dataset.select(keep_idx)`, not `.filter()`, since
  `.select()` doesn't re-scan every large token row).
- The model bundle's `model_config.yaml` fixes `n_neighs: 10` and
  `enc_emb_dim: 384` — exactly matching this task's "k=10" and the expected
  `(10000, 384)` output shape. We don't set these ourselves; they come from
  the pretrained model's own config, confirmed below.
- Bundle files `ensembl_dictionary.pkl` / `gene_count_dictionary.pkl` are
  passed to `harmonize_adata` explicitly, mirroring exactly what
  `harmonize_tokenize_embed_pipeline` does internally when these paths aren't
  given, so harmonization matches how the model was trained.

In [ ]:
import yaml
from terra import download_pretrained, harmonize_adata, tokenize_adata, embed_dataset

model_dir = Path(download_pretrained("Lotfollahi-lab/TERRA-112M"))
print("model_dir:", model_dir)

with open(model_dir / "model_config.yaml") as f:
    model_config = yaml.safe_load(f)
print("n_neighs:", model_config["data"]["n_neighs"])
print("embedding dim (enc_emb_dim):", model_config["meta"]["enc_emb_dim"])
print("tokenizer_type:", model_config["data"]["tokenizer_type"])

### 2a. k=10 spatial neighbor graph on the full section

Built with the same call TERRA's own tokenizer uses internally
(`squidpy.gr.spatial_neighbors`, `coord_type='generic'`, `n_neighs=10`), on
**all 365k+ cells** in `adata_full` -- not the 10,000-cell subsample -- so
neighborhoods reflect real tissue context. We visualize a small window so the
graph structure is checkable by eye.

In [ ]:
sq.gr.spatial_neighbors(
    adata_full, coord_type="generic", spatial_key="spatial", n_neighs=10
)
print(
    adata_full.obsp["spatial_connectivities"].shape,
    "edges:",
    adata_full.obsp["spatial_connectivities"].nnz,
)

In [ ]:
# Visualize the k=10 graph in a small 150x150 um window (native pixel size
# 0.2125 um/px), drawn directly on top of the H&E patch it corresponds to.
# Read via tifffile/zarr -- these HEST WSIs are generic pyramidal TIFFs, so a
# small region can be pulled straight from level 0 without OpenSlide/pyvips
# (not installed in this kernel; see the environment note at the top).
import tifffile
import zarr

pixel_size = 0.2125
window_um = 150
window_px = int(round(window_um / pixel_size))

xy = adata_full.obsm["spatial"]
center = np.median(xy, axis=0)
x0 = int(center[0] - window_px / 2)
y0 = int(center[1] - window_px / 2)

in_window = (
    (xy[:, 0] >= x0)
    & (xy[:, 0] < x0 + window_px)
    & (xy[:, 1] >= y0)
    & (xy[:, 1] < y0 + window_px)
)
idx_in_window = np.where(in_window)[0]

wsi_store = tifffile.imread(
    HEST_DIR / "wsis" / f"{SAMPLE_ID}.tif", aszarr=True, level=0
)
wsi_level0 = zarr.open(wsi_store, mode="r")
patch = wsi_level0[
    y0 : y0 + window_px, x0 : x0 + window_px
]  # zarr axes are (row=y, col=x, channel)

conn = adata_full.obsp["spatial_connectivities"].tocsr()

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(patch)
for i in idx_in_window:
    neighbors = conn[i].indices
    for j in neighbors:
        if in_window[j]:
            ax.plot(
                [xy[i, 0] - x0, xy[j, 0] - x0],
                [xy[i, 1] - y0, xy[j, 1] - y0],
                c="black",
                linewidth=0.7,
                zorder=1,
            )
ax.scatter(
    xy[idx_in_window, 0] - x0, xy[idx_in_window, 1] - y0, s=10, c="yellow", zorder=2
)
ax.axis("off")
ax.set_title(
    f"k=10 spatial neighbor graph on H&E ({window_um}x{window_um} um, {len(idx_in_window)} cells)"
)
fig.tight_layout()

### 2b. Harmonize and tokenize the full section

`harmonize_adata` maps our real gene symbols to the model's Ensembl-ID
vocabulary and validates raw integer counts; `tokenize_adata` builds each
cell's tokenized neighborhood (using the `n_neighs=10` graph above) for
**every** cell in the full section. This is the CPU-bound step -- only the
final `embed_dataset` call needs the GPU, and we restrict that one to the
10,000 sampled cells.

In [ ]:
gene_mapping_dict_file_path = model_dir / "ensembl_dictionary.pkl"
gene_occurrence_count_file_path = model_dir / "gene_count_dictionary.pkl"
assert gene_mapping_dict_file_path.is_file()
assert gene_occurrence_count_file_path.is_file()

harmonized_full = harmonize_adata(
    adata_full,
    gene_mapping_dict_file_path=str(gene_mapping_dict_file_path),
    gene_occurrence_count_file_path=str(gene_occurrence_count_file_path),
    min_cells_per_gene=0,
    min_genes_per_cell=0,
)
print(harmonized_full)
print(f"Genes kept after harmonization: {harmonized_full.n_vars} / {adata_full.n_vars}")
print(f"Cells kept after harmonization: {harmonized_full.n_obs} / {adata_full.n_obs}")

In [ ]:
import logging

logging.basicConfig(level="INFO")

full_dataset = tokenize_adata(
    adata=harmonized_full,
    model_folder_path=str(model_dir),
    cache_directory_path=str(CACHE_DIR),
    nproc=2,
)
print(full_dataset)

### 2c. Restrict to the 10,000 sampled cells, then embed

Same pattern as `downstream_analysis.ipynb`'s subset-analysis section: read
`cell_id` once, build row indices, `.select()` (not `.filter()`, which would
re-scan every large token row just to test one field).

In [ ]:
sampled_ids_str = set(map(str, sampled_cell_ids))
keep_idx = [
    i for i, cid in enumerate(full_dataset["cell_id"]) if str(cid) in sampled_ids_str
]
assert len(keep_idx) == N_CELLS, f"expected {N_CELLS} matches, got {len(keep_idx)}"

subset_dataset = full_dataset.select(keep_idx)
subset_dataset

In [ ]:
embeddings = embed_dataset(
    dataset=subset_dataset,
    model_folder_path=str(model_dir),
    batch_size=32,
)
print(list(embeddings))
for key, arr in embeddings.items():
    print(key, arr.shape)

### 2d. Save embeddings

We save both `cell_emb` and `neighborhood_emb` (whichever `terra-uni2` fusion
step follows can pick the right one) alongside the exact cell IDs in the row
order `embed_dataset` produced -- **not** `sampled_cell_ids` from Step 1,
since `.select()` may reorder rows; we re-read `cell_id` from
`subset_dataset` to guarantee the saved order matches the embedding rows.

In [ ]:
ordered_cell_ids = np.array(subset_dataset["cell_id"])

np.savez(
    EMB_DIR / f"{SAMPLE_ID}_terra.npz",
    cell_ids=ordered_cell_ids,
    cell_emb=embeddings["cell_emb"],
    neighborhood_emb=embeddings["neighborhood_emb"],
)
print(f"Saved -> {EMB_DIR / f'{SAMPLE_ID}_terra.npz'}")
print("cell_emb shape:", embeddings["cell_emb"].shape)
print("neighborhood_emb shape:", embeddings["neighborhood_emb"].shape)

### 2e. Sanity check: are the embeddings informative?

Per the zero-shot quickstart's own sanity check: a very high average cosine
similarity (~1) means the embeddings collapsed. Then UMAP both embeddings,
colored by `total_counts` and by 2-3 top highly-variable genes -- clear
structure is the pass condition; noise means stop and report back before
moving to UNI2-h.

In [ ]:
for emb_key in ["cell_emb", "neighborhood_emb"]:
    X = embeddings[emb_key]
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
    n = Xn.shape[0]
    S = Xn.sum(axis=0)
    avg_cos = (S @ S - n) / (n * (n - 1))
    print(f"{emb_key}: average cosine similarity = {avg_cos:.3f}")

In [ ]:
# Build a small AnnData over just the sampled cells to carry QC + HVG info for
# the UMAP coloring below (order-aligned to `ordered_cell_ids`).
adata_sub = adata_ok[[str(c) for c in ordered_cell_ids]].copy()

sc.pp.normalize_total(adata_sub)
sc.pp.log1p(adata_sub)
sc.pp.highly_variable_genes(adata_sub, n_top_genes=3)
top_hvgs = adata_sub.var_names[adata_sub.var["highly_variable"]].tolist()
print("Top HVGs used for coloring:", top_hvgs)

for emb_key in ["cell_emb", "neighborhood_emb"]:
    adata_sub.obsm[emb_key] = embeddings[emb_key]

In [ ]:
for emb_key in ["cell_emb", "neighborhood_emb"]:
    sc.pp.neighbors(adata_sub, use_rep=emb_key, key_added=emb_key)
    sc.tl.umap(adata_sub, neighbors_key=emb_key)
    sc.pl.umap(
        adata_sub,
        neighbors_key=emb_key,
        color=["total_counts"] + top_hvgs,
        title=[f"{emb_key}: total_counts"] + [f"{emb_key}: {g}" for g in top_hvgs],
    )

If the UMAPs above show clear structure (cells organizing by total counts
and/or marker gene expression, not a uniform blob), TERRA's embeddings are
informative for this sample and Step 3 (UNI2-h extraction) can proceed. If
they look like noise, stop here before spending compute on UNI2-h and the
linear probes.